In [ ]:
import numpy as np, json, random, warnings
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import r2_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr, spearmanr, kendalltau as kt
import pandas as pd
warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_ROOT = Path("/kaggle/working/phase3_comparison")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════
# LOAD BOTH EMBEDDING FILES
# ═══════════════════════════════════════════════════════════
# Two embeddings to compare:
#   1. "ViT-Base"    = extracted from base SwinUNETR (before triplet training)
#   2. "ViT-Triplet" = extracted from v4 SwinUNETR (after triplet training)
#
# Both use the SAME 4233-D extraction (octant+region+vol).
# The only difference is the encoder weights.
#
# HOW TO SET UP:
#   Upload both .npz files as Kaggle datasets with DIFFERENT names.
#   Adjust the filenames below if needed.
# ═══════════════════════════════════════════════════════════

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_npz(patterns):
    """Search for an .npz file matching any of the patterns."""
    for pat in patterns:
        for root in SEARCH_ROOTS:
            matches = list(root.rglob(f"{pat}.npz"))
            if matches:
                return matches[0]
    return None

def load_embedding(path):
    """Load an .npz embedding file → dict of {pid__tp: vector}."""
    data = np.load(path, allow_pickle=True)
    embs_arr = data["embeddings"]
    pids_arr = data["patient_ids"]
    tps_arr  = data["timepoints"]
    emb_dict = {}
    for i in range(len(embs_arr)):
        key = f"{pids_arr[i]}__{tps_arr[i]}"
        emb_dict[key] = embs_arr[i]
    return emb_dict, embs_arr.shape[1]

# ── Find embedding files ──
# ViT-Base: the extraction from BEFORE triplet training
# Possible filenames: vit_swinunetr_embeddings.npz or vit_swinunetr_embeddings_v3_base.npz
base_path = find_npz(["vit_swinunetr_embeddings_v3_base",
                       "vit_swinunetr_embeddings_base",
                       "vit_swinunetr_embeddings"])

# ViT-Triplet: the extraction from AFTER triplet training (v4 model)
# Possible filenames: vit_swinunetr_embeddings_v3.npz (from extract-only run)
tri_path = find_npz(["vit_swinunetr_embeddings_v3",
                      "vit_swinunetr_embeddings_v4",
                      "vit_swinunetr_embeddings_triplet"])

models = {}

if base_path:
    embs_base, dim_base = load_embedding(base_path)
    models["ViT-Base"] = embs_base
    print(f"ViT-Base:    {len(embs_base)} scans | dim={dim_base} | {base_path}")
else:
    print("⚠ ViT-Base embeddings not found!")

if tri_path:
    embs_tri, dim_tri = load_embedding(tri_path)
    models["ViT-Triplet"] = embs_tri
    print(f"ViT-Triplet: {len(embs_tri)} scans | dim={dim_tri} | {tri_path}")
else:
    print("⚠ ViT-Triplet embeddings not found!")

if base_path and tri_path and str(base_path) == str(tri_path):
    print("\n⚠ WARNING: Both models loaded the SAME file!")
    print("  You need TWO different .npz files:")
    print("  1. Base model extraction (before triplet training)")
    print("  2. Triplet model extraction (after triplet training)")
    print("  Rename one to vit_swinunetr_embeddings_v3_base.npz")

if not models:
    raise FileNotFoundError("No embeddings found!")

print(f"\nModels loaded: {list(models.keys())}")

# ── Load tumour volume metadata ──
tumor_df = None
for f in (list(Path("/kaggle/input").rglob("tumor_volumes.csv")) +
          list(Path("/kaggle/working").rglob("tumor_volumes.csv"))):
    tumor_df = pd.read_csv(f); break

print(f"Tumor volumes: {'loaded' if tumor_df is not None else 'NOT FOUND'}")

# ── Sanity check ──
for mn, embs in models.items():
    keys = list(embs.keys())
    arr = np.stack([embs[k] for k in keys])
    norms = np.linalg.norm(arr, axis=1)
    pids = set(k.split("__")[0] for k in keys)
    print(f"  {mn}: {len(keys)} scans | {len(pids)} patients | "
          f"norm=[{norms.min():.1f}, {norms.max():.1f}] mean={norms.mean():.1f}")

# ═══════════════════════════════════════════════════════════
# COMPONENT-WISE L2 NORMALISATION (same for all models)
# ═══════════════════════════════════════════════════════════
for mn in list(models.keys()):
    emb_dict = models[mn]
    keys = list(emb_dict.keys())
    arr = np.stack([emb_dict[k] for k in keys])
    D = arr.shape[1]

    if D >= 2121:
        # Octant + Region + Vol format
        comp_octant = arr[:, 0:D-576-9]
        comp_region = arr[:, D-576-9:D-9]
        comp_vol    = arr[:, D-9:]

        comp_octant_n = comp_octant / (np.linalg.norm(comp_octant, axis=1, keepdims=True) + 1e-8)
        comp_region_n = comp_region / (np.linalg.norm(comp_region, axis=1, keepdims=True) + 1e-8)
        comp_vol_n    = comp_vol    / (np.linalg.norm(comp_vol,    axis=1, keepdims=True) + 1e-8)
        arr_balanced = np.concatenate([comp_octant_n, comp_region_n, comp_vol_n * 2], axis=1)
        models[mn] = {k: arr_balanced[i] for i, k in enumerate(keys)}
        print(f"  {mn}: Component-wise L2 (D={D} → balanced {arr_balanced.shape[1]})")
    else:
        arr_n = arr / (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-8)
        models[mn] = {k: arr_n[i] for i, k in enumerate(keys)}
        print(f"  {mn}: Global L2 normalisation (D={D})")

# Match helper
def match_row(tumor_df, pid, tp):
    if tumor_df is None: return None
    m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                 (tumor_df["timepoint"].astype(str) == str(tp))]
    if len(m) > 0: return m.iloc[0]
    tp_int = int(tp) - 100 if str(tp).isdigit() and int(tp) >= 100 else int(tp)
    m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                 (tumor_df["timepoint"].astype(str) == str(tp_int))]
    if len(m) > 0: return m.iloc[0]
    return None

results = {}
print("\n✅ Ready for evaluation")


In [ ]:
# ═══════════════════════════════════════════════════════════
# FULL 18-TEST BATTERY — computed for ALL models (no hardcoding)
# ═══════════════════════════════════════════════════════════
print("=" * 70)
print("  COMPUTING ALL 18 TESTS FOR EACH MODEL")
print("=" * 70)

for mn, embs in models.items():
    print(f"\n{'─'*50}")
    print(f"  Model: {mn}")
    print(f"{'─'*50}")
    results[mn] = {}
    keys = list(embs.keys())
    X = np.stack([embs[k] for k in keys])
    X_l2 = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    Xs = StandardScaler().fit_transform(X_l2)

    # ── M1-M6: MORPHOLOGY ──
    mX, mvols = [], {"wt": [], "tc": [], "et": []}
    for k in keys:
        pid, tp = k.split("__")
        row = match_row(tumor_df, pid, tp)
        if row is not None:
            mX.append(Xs[keys.index(k)])
            for r_name in ["wt", "tc", "et"]:
                for col in [f"{r_name}_vol", f"{r_name.upper()}_vol",
                             f"{r_name}_volume", f"vol_{r_name}"]:
                    if col in row.index:
                        mvols[r_name].append(float(row[col])); break
                else:
                    mvols[r_name].append(0.0)

    print(f"  Matched: {len(mX)}/{len(keys)} scans to tumor_df")

    if len(mX) >= 10:
        mX = np.stack(mX)
        y_wt = np.array(mvols["wt"])
        ridge = Ridge(alpha=1.0)
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

        # M1: Volume
        p_ridge = cross_val_predict(ridge, mX, y_wt, cv=5)
        p_rf = cross_val_predict(rf, mX, y_wt, cv=5)
        results[mn]["M1_volume_R2_ridge"] = float(r2_score(y_wt, p_ridge))
        results[mn]["M1_volume_R2_rf"] = float(r2_score(y_wt, p_rf))
        rho, _ = spearmanr(y_wt, p_rf)
        results[mn]["M1_spearman_rho"] = abs(float(rho))

        # M2: LogVol
        y_log = np.log1p(y_wt)
        p2r = cross_val_predict(ridge, mX, y_log, cv=5)
        p2f = cross_val_predict(rf, mX, y_log, cv=5)
        results[mn]["M2_logvol_R2_ridge"] = float(r2_score(y_log, p2r))
        results[mn]["M2_logvol_R2_rf"] = float(r2_score(y_log, p2f))

        # M3: Enhancement ratio
        y_enh = np.array(mvols["et"]) / (np.array(mvols["wt"]) + 1e-6)
        p3r = cross_val_predict(ridge, mX, y_enh, cv=5)
        p3f = cross_val_predict(rf, mX, y_enh, cv=5)
        results[mn]["M3_enhancement_ridge"] = float(r2_score(y_enh, p3r))
        results[mn]["M3_enhancement_rf"] = float(r2_score(y_enh, p3f))

        # M4: Necrosis (binary: has TC > 0?)
        y_nec = (np.array(mvols["tc"]) > 0.1).astype(int)
        if len(set(y_nec)) > 1:
            f1s = cross_val_score(LogisticRegression(max_iter=500), mX, y_nec, cv=5, scoring="f1")
            results[mn]["M4_necrosis_F1"] = float(np.mean(f1s))
        else:
            results[mn]["M4_necrosis_F1"] = 0.0

        # M5: Core fraction
        y_cf = np.array(mvols["tc"]) / (np.array(mvols["wt"]) + 1e-6)
        p5r = cross_val_predict(ridge, mX, y_cf, cv=5)
        p5f = cross_val_predict(rf, mX, y_cf, cv=5)
        results[mn]["M5_corefrac_ridge"] = float(r2_score(y_cf, p5r))
        results[mn]["M5_corefrac_rf"] = float(r2_score(y_cf, p5f))

    # M6: Patient purity (label-free — always computable)
    pids_arr = np.array([k.split("__")[0] for k in keys])
    nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
    _, idx = nbrs.kneighbors(Xs)
    results[mn]["M6_patient_purity_pct"] = float(
        100 * np.mean([np.mean(pids_arr[idx[i, 1:]] == pids_arr[i])
                       for i in range(len(keys))]))

    # ── H1-H5: HETEROGENEITY ──
    U, S, Vh = np.linalg.svd(Xs, full_matrices=False)
    p_sv = S / S.sum()
    rankme = float(np.exp(-np.sum(p_sv * np.log(p_sv + 1e-12))))
    eff_rank_95 = int(np.searchsorted(np.cumsum(p_sv), 0.95)) + 1
    results[mn]["H1_rankme"] = rankme
    results[mn]["H1_eff_rank_95"] = float(eff_rank_95)
    results[mn]["H5_rankme_standalone"] = rankme

    emb_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    idx2 = np.random.choice(len(keys), (2000, 2), replace=True)
    sims = (emb_n[idx2[:, 0]] * emb_n[idx2[:, 1]]).sum(1)
    results[mn]["H2_diversity"] = float(1 - np.mean(sims))
    sq_dist = np.sum((emb_n[idx2[:, 0]] - emb_n[idx2[:, 1]])**2, axis=1)
    results[mn]["H2_uniformity"] = float(np.log(np.mean(np.exp(-2 * sq_dist)) + 1e-10))

    # H3: Responder F1 (volume change > 20%)
    if len(mX) >= 10:
        pe_h3 = {}
        for k in keys:
            pid, tp = k.split("__")
            if pid not in pe_h3: pe_h3[pid] = {}
            pe_h3[pid][tp] = k
        resp_X, resp_y = [], []
        for pid, tps_dict in pe_h3.items():
            stps = sorted(tps_dict.keys())
            if len(stps) < 2: continue
            k0, kT = tps_dict[stps[0]], tps_dict[stps[-1]]
            r0 = match_row(tumor_df, pid, stps[0])
            rT = match_row(tumor_df, pid, stps[-1])
            if r0 is not None and rT is not None:
                wt0 = float(r0.get("wt_vol", 0) if hasattr(r0, "get") else r0["wt_vol"])
                wtT = float(rT.get("wt_vol", 0) if hasattr(rT, "get") else rT["wt_vol"])
                if wt0 > 0.1:
                    resp_X.append(Xs[keys.index(k0)])
                    resp_y.append(int((wtT - wt0) / wt0 > 0.20))
        if len(resp_X) >= 10 and len(set(resp_y)) > 1:
            resp_X = np.stack(resp_X)
            f1s = cross_val_score(LogisticRegression(max_iter=500),
                                  resp_X, np.array(resp_y), cv=5, scoring="f1")
            results[mn]["H3_responder_F1"] = float(np.mean(f1s))
        else:
            results[mn]["H3_responder_F1"] = 0.0
    else:
        results[mn]["H3_responder_F1"] = 0.0

    norms_final = np.linalg.norm(X_l2, axis=1)
    results[mn]["H4_norm_cv"] = float(np.std(norms_final) / (np.mean(norms_final) + 1e-8))

    # ── T1-T8: TEMPORAL ──
    raw = np.stack([embs[k] for k in keys])
    norms_l2 = np.linalg.norm(raw, axis=1, keepdims=True) + 1e-8
    embs_l2 = {k: embs[k] / norms_l2[i] for i, k in enumerate(keys)}
    pe = {}
    for k in keys:
        pid, tp = k.split("__")
        if pid not in pe: pe[pid] = {}
        pe[pid][tp] = embs[k]
    longi = {p: t for p, t in pe.items() if len(t) >= 2}
    print(f"  Longitudinal patients (2+ scans): {len(longi)}")

    den, dvol_wt, dvol_et, csim = [], [], [], []
    v_first = []

    for pid, tps_dict in longi.items():
        stps = sorted(tps_dict.keys())
        e_first = embs_l2[f"{pid}__{stps[0]}"]
        e_last = embs_l2[f"{pid}__{stps[-1]}"]
        v_first.append(np.linalg.norm(e_last - e_first))

        for ii in range(len(stps) - 1):
            e0 = embs_l2[f"{pid}__{stps[ii]}"]
            e1 = embs_l2[f"{pid}__{stps[ii+1]}"]
            d = np.linalg.norm(e1 - e0)
            den.append(d)
            n0 = np.linalg.norm(e0) + 1e-8
            n1 = np.linalg.norm(e1) + 1e-8
            csim.append(float((e0/n0) @ (e1/n1)))
            if tumor_df is not None:
                v0 = tumor_df[(tumor_df["patient_id"]==pid) & (tumor_df["timepoint"]==int(stps[ii]))]
                v1 = tumor_df[(tumor_df["patient_id"]==pid) & (tumor_df["timepoint"]==int(stps[ii+1]))]
                if len(v0) > 0 and len(v1) > 0:
                    dvol_wt.append(abs(v1.iloc[0]["wt_vol"] - v0.iloc[0]["wt_vol"]))
                    et0 = v0.iloc[0]["et_vol"]; et1 = v1.iloc[0]["et_vol"]
                    dvol_et.append(et1 / (et0 + 1e-6) - 1)

    den = np.array(den)
    ml = min(len(den), len(dvol_wt))

    # T1
    if ml >= 5:
        rho_wt, _ = spearmanr(den[:ml], dvol_wt[:ml])
        results[mn]["T1_spearman_wt"] = abs(float(rho_wt))
    else:
        results[mn]["T1_spearman_wt"] = 0

    # T2
    near_dup = np.mean(den < 0.001 * den.mean()) if len(den) > 0 else 1.0
    results[mn]["T2_ordering_pass"] = float(near_dup < 0.01)

    # T3
    ml_et = min(len(den), len(dvol_et))
    if ml_et >= 10:
        yd = np.array(dvol_wt[:ml_et])
        pd3 = cross_val_predict(Ridge(1.0), den[:ml_et].reshape(-1,1), yd, cv=min(5, ml_et//2))
        results[mn]["T3_delta_R2"] = float(r2_score(yd, pd3))
        vol_sign = (np.array(dvol_wt[:ml_et]) > 0).astype(int)
        drift_sign = (den[:ml_et] > np.median(den[:ml_et])).astype(int)
        if len(set(vol_sign)) > 1:
            results[mn]["T3_directional_auc"] = float(roc_auc_score(vol_sign, drift_sign))
        else:
            results[mn]["T3_directional_auc"] = 0.5
    else:
        results[mn]["T3_delta_R2"] = 0
        results[mn]["T3_directional_auc"] = 0.5

    # T4 — RANO
    if ml_et >= 10 and tumor_df is not None:
        progressive = (np.array(dvol_et[:ml_et]) > 0.40).astype(int)
        if len(set(progressive)) > 1:
            results[mn]["T4_rano_auc"] = float(roc_auc_score(progressive, den[:ml_et]))
        else:
            results[mn]["T4_rano_auc"] = 0.5
    else:
        results[mn]["T4_rano_auc"] = 0.5

    # T5
    coherence = float(np.mean(csim)) if csim else 0
    results[mn]["T5_coherence"] = coherence
    results[mn]["T5_pass_dual"] = float(0.70 < coherence < 0.93)

    # T6
    results[mn]["T6_velocity_cv"] = float(np.std(den) / (np.mean(den) + 1e-8)) if len(den) else 0

    # T7
    all_dists = np.array(v_first)
    if tumor_df is not None and len(all_dists) >= 10:
        pids_longi = list(longi.keys())
        prog_mask = []
        for pid in pids_longi:
            stps = sorted(longi[pid].keys())
            v0 = tumor_df[(tumor_df["patient_id"]==pid) & (tumor_df["timepoint"]==int(stps[0]))]
            vT = tumor_df[(tumor_df["patient_id"]==pid) & (tumor_df["timepoint"]==int(stps[-1]))]
            if len(v0) > 0 and len(vT) > 0:
                prog_mask.append((vT.iloc[0]["et_vol"] / (v0.iloc[0]["et_vol"] + 1e-6) - 1) > 0.40)
            else:
                prog_mask.append(False)
        prog_mask = np.array(prog_mask)
        d_prog = all_dists[prog_mask]; d_stab = all_dists[~prog_mask]
        if len(d_prog) >= 3 and len(d_stab) >= 3:
            pooled = np.sqrt((np.var(d_prog) + np.var(d_stab)) / 2) + 1e-8
            results[mn]["T7_treatment_d"] = float(abs(d_prog.mean() - d_stab.mean()) / pooled)
        else:
            results[mn]["T7_treatment_d"] = 0
    else:
        results[mn]["T7_treatment_d"] = 0

    # T8 — Kendall tau (FIXED: use 2+ timepoints, not 3+)
    taus = []
    for pid, tps_dict in longi.items():
        stps = sorted(tps_dict.keys())
        if len(stps) < 2: continue  # FIXED: was < 3
        e_base = embs_l2[f"{pid}__{stps[0]}"]
        dists = [np.linalg.norm(embs_l2[f"{pid}__{v}"] - e_base) for v in stps[1:]]
        if len(dists) >= 2:
            tau, _ = kt(dists, range(len(dists)))
            if not np.isnan(tau):
                taus.append(tau)
        elif len(dists) == 1:
            # Only 2 timepoints: distance is either positive (good) or not
            # Can't compute Kendall tau with 1 point, skip gracefully
            pass
    results[mn]["T8_kendall_tau"] = float(np.mean(taus)) if taus else float("nan")
    n_t8 = len(taus)
    print(f"  T8: {n_t8} patients with 3+ scans for Kendall tau")

    # Print all results for this model
    for k in sorted(results[mn].keys()):
        v = results[mn][k]
        print(f"    {k:35s} = {v:.4f}" if not np.isnan(v) else f"    {k:35s} = nan")

print("\n✅ All tests computed")


In [ ]:
# ═══════════════════════════════════════════════════════════
# COMPARISON TABLE — ViT-Base vs ViT-Triplet (ALL computed, nothing hardcoded)
# ═══════════════════════════════════════════════════════════
print("=" * 80)
print("  PRE vs POST TRIPLET TRAINING — HEAD-TO-HEAD COMPARISON")
print("=" * 80)

THRESHOLDS = {
    "M1_volume_R2_ridge":    (0.50, "low"),
    "M1_volume_R2_rf":       (0.50, "med"),
    "M1_spearman_rho":       (0.55, "med"),
    "M2_logvol_R2_ridge":    (0.40, "low"),
    "M2_logvol_R2_rf":       (0.40, "med"),
    "M3_enhancement_ridge":  (0.25, "low"),
    "M3_enhancement_rf":     (0.25, "low"),
    "M4_necrosis_F1":        (0.60, "med"),
    "M5_corefrac_ridge":     (0.30, "low"),
    "M5_corefrac_rf":        (0.30, "low"),
    "M6_patient_purity_pct": (60.0, "HIGH"),
    "H1_rankme":             (30.0, "med"),
    "H1_eff_rank_95":        (50.0, "med"),
    "H2_diversity":          (0.25, "med"),
    "H2_uniformity":         (-3.0, "med"),
    "H3_responder_F1":       (0.55, "med"),
    "H4_norm_cv":            (0.30, "low"),
    "H5_rankme_standalone":  (30.0, "med"),
    "T1_spearman_wt":        (0.30, "HIGH"),
    "T2_ordering_pass":      (1.0,  "low"),
    "T3_delta_R2":           (None, "desc"),
    "T3_directional_auc":    (0.55, "low"),
    "T4_rano_auc":           (0.65, "HIGH"),
    "T5_coherence":          (0.70, "med"),
    "T5_pass_dual":          (1.0,  "low"),
    "T6_velocity_cv":        (None, "desc"),
    "T7_treatment_d":        (0.50, "HIGH"),
    "T8_kendall_tau":        (0.30, "HIGH"),
}

# Get all model names
model_names = list(results.keys())
all_metrics = sorted(set().union(*[set(r.keys()) for r in results.values()]))

# Header
header = f"{'Metric':40s}"
for mn in model_names:
    header += f"  {mn:>12s}"
header += "   Winner    Priority"
print(header)
print("─" * len(header))

wins = {mn: 0 for mn in model_names}
n_pass = {mn: 0 for mn in model_names}

categories = {
    "M": "── MORPHOLOGY ──",
    "H": "── HETEROGENEITY ──",
    "T": "── TEMPORAL ──",
}

prev_cat = ""
for metric in all_metrics:
    cat = metric[0]
    if cat != prev_cat and cat in categories:
        print(f"\n{categories[cat]}")
        prev_cat = cat

    thresh_info = THRESHOLDS.get(metric, (None, ""))
    thresh, priority = thresh_info if thresh_info else (None, "")

    line = f"  {metric:38s}"
    vals = []
    for mn in model_names:
        v = results[mn].get(metric, float("nan"))
        vals.append(v)
        if np.isnan(v):
            line += f"  {'nan':>12s}"
        else:
            line += f"  {v:>12.4f}"

    # Determine winner (higher is better for all except T5_pass_dual which is binary)
    if len(vals) == 2 and not any(np.isnan(v) for v in vals):
        if vals[0] > vals[1]:
            line += f"   ← {model_names[0]:10s}"
            wins[model_names[0]] += 1
        elif vals[1] > vals[0]:
            line += f"   → {model_names[1]:10s}"
            wins[model_names[1]] += 1
        else:
            line += f"   = tie"
    else:
        line += f"   {'?':>12s}"

    # Priority + pass/fail
    if thresh is not None and priority != "desc":
        line += f"  [{priority}]"
        for j, mn in enumerate(model_names):
            v = vals[j]
            if not np.isnan(v) and v >= thresh:
                n_pass[mn] += 1
    elif priority == "desc":
        line += f"  [desc]"

    print(line)

# Summary
print(f"\n{'═' * 80}")
print(f"  SUMMARY")
print(f"{'═' * 80}")
for mn in model_names:
    print(f"  {mn}: {n_pass[mn]} tests pass | {wins[mn]} metric wins")

if len(model_names) == 2:
    m0, m1 = model_names
    print(f"\n  {m0} wins {wins[m0]} metrics")
    print(f"  {m1} wins {wins[m1]} metrics")
    if wins[m1] > wins[m0]:
        print(f"\n  → Triplet training improved {wins[m1] - wins[m0]} more metrics than it hurt")
    elif wins[m0] > wins[m1]:
        print(f"\n  → Base model was better on {wins[m0] - wins[m1]} more metrics")
    else:
        print(f"\n  → Tie")

    # HIGH-PRI comparison
    print(f"\n  HIGH-PRIORITY METRICS:")
    for metric in all_metrics:
        t = THRESHOLDS.get(metric)
        if t and t[1] == "HIGH":
            v0 = results[m0].get(metric, float("nan"))
            v1 = results[m1].get(metric, float("nan"))
            delta = v1 - v0 if not (np.isnan(v0) or np.isnan(v1)) else float("nan")
            arrow = "✅ ↑" if delta > 0 else ("❌ ↓" if delta < 0 else "=")
            print(f"    {metric:35s} Base={v0:>8.4f}  Triplet={v1:>8.4f}  "
                  f"Δ={delta:>+8.4f}  {arrow}" if not np.isnan(delta) else
                  f"    {metric:35s} Base={v0}  Triplet={v1}  Δ=nan")

# Save results
with open(OUTPUT_ROOT / "comparison_results.json", "w") as f:
    json.dump({mn: {k: float(v) if not np.isnan(v) else None
                    for k, v in r.items()}
               for mn, r in results.items()}, f, indent=2)
print(f"\nSaved: {OUTPUT_ROOT / 'comparison_results.json'}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# t-SNE VISUALIZATION — side by side, colored by patient
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, len(models), figsize=(8*len(models), 7))
if len(models) == 1: axes = [axes]

for ax_i, (mn, embs) in enumerate(models.items()):
    keys = list(embs.keys())
    X = np.stack([embs[k] for k in keys])
    pids = [k.split("__")[0] for k in keys]

    # Only show patients with 2+ scans (longitudinal)
    from collections import Counter
    pid_counts = Counter(pids)
    multi_pids = {p for p, c in pid_counts.items() if c >= 2}

    # Subsample for speed (max 500 scans)
    mask = [i for i, p in enumerate(pids) if p in multi_pids]
    if len(mask) > 500:
        mask = sorted(random.sample(mask, 500))

    X_sub = X[mask]
    pids_sub = [pids[i] for i in mask]

    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    emb_2d = tsne.fit_transform(X_sub)

    # Color by patient (top 20 most frequent patients get distinct colors)
    pid_counts_sub = Counter(pids_sub)
    top_pids = [p for p, _ in pid_counts_sub.most_common(20)]
    colors = plt.cm.tab20(np.linspace(0, 1, 20))

    for j, pid in enumerate(top_pids):
        idx_p = [i for i, p in enumerate(pids_sub) if p == pid]
        axes[ax_i].scatter(emb_2d[idx_p, 0], emb_2d[idx_p, 1],
                          c=[colors[j]], label=pid.split("-")[-1],
                          s=30, alpha=0.7)
        # Draw trajectory lines
        if len(idx_p) >= 2:
            tps = [keys[mask[i]].split("__")[1] for i in idx_p]
            order = np.argsort(tps)
            pts = emb_2d[np.array(idx_p)[order]]
            axes[ax_i].plot(pts[:, 0], pts[:, 1], c=colors[j], alpha=0.3, linewidth=1)

    # Gray dots for other patients
    other_idx = [i for i, p in enumerate(pids_sub) if p not in top_pids]
    if other_idx:
        axes[ax_i].scatter(emb_2d[other_idx, 0], emb_2d[other_idx, 1],
                          c="lightgray", s=10, alpha=0.3)

    purity = results[mn].get("M6_patient_purity_pct", 0)
    axes[ax_i].set_title(f"{mn}\nM6 Purity={purity:.1f}%", fontsize=14, fontweight="bold")
    axes[ax_i].set_xlabel("t-SNE 1"); axes[ax_i].set_ylabel("t-SNE 2")

plt.tight_layout()
fig_path = OUTPUT_ROOT / "tsne_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {fig_path}")
